# Day 15 — Feature Engineering 🚀

Feature engineering creates useful features from raw data to help a machine learning model learn better patterns.

## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

## 2. Load Titanic Dataset

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv')

print('Dataset Shape:', df.shape)
display(df.head())

## 3. Check Missing Values

In [ ]:
print(df.isnull().sum())

## 4. Feature Engineering

Create `FamilySize`, `IsAlone`, `FarePerPerson`, `Title`, and `AgeGroup`.

In [ ]:
df['FamilySize'] = df['sibsp'] + df['parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['FarePerPerson'] = df['fare'] / df['FamilySize']
df['Title'] = df['who'].astype(str).str.title()
df['AgeGroup'] = pd.cut(df['age'], bins=[0,12,18,35,60,100], labels=['Child','Teen','Young Adult','Adult','Senior'])

display(df[['FamilySize','IsAlone','FarePerPerson','Title','AgeGroup']].head(10))

## 5. Analyze Engineered Features

In [ ]:
print('Family Size Distribution:')
print(df['FamilySize'].value_counts().sort_index())

print('\nAge Group Distribution:')
print(df['AgeGroup'].value_counts().sort_index())

print('\nTitle Distribution:')
print(df['Title'].value_counts())

## 6. Visualize Survival Rate by Family Size

In [ ]:
plt.figure(figsize=(7,5))
df.groupby('FamilySize', observed=True)['survived'].mean().head(8).plot(kind='bar')
plt.xlabel('Family Size')
plt.ylabel('Survival Rate')
plt.title('Survival Rate by Family Size')
plt.tight_layout()
plt.show()

## 7. Select Features and Target

In [ ]:
features = ['pclass','sex','age','fare','embarked','FamilySize','IsAlone','FarePerPerson','Title','AgeGroup']
X = df[features]
y = df['survived']

numeric_features = ['pclass','age','fare','FamilySize','IsAlone','FarePerPerson']
categorical_features = ['sex','embarked','Title','AgeGroup']

## 8. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print('Training Samples:', len(X_train))
print('Testing Samples :', len(X_test))

## 9. Preprocessing Pipeline

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

## 10. Logistic Regression Model

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Model training and prediction completed.')

## 11. Model Evaluation

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

## 12. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Survived','Survived'])
disp.plot()
plt.title('Feature Engineering - Confusion Matrix')
plt.tight_layout()
plt.show()

## 13. Final Summary

In [ ]:
engineered_features = ['FamilySize','IsAlone','FarePerPerson','Title','AgeGroup']

print('=' * 60)
print('DAY 15 - FINAL SUMMARY')
print('=' * 60)
print(f'Dataset Shape  : {df.shape}')
print(f'Model Accuracy : {accuracy:.4f}')
print('\nEngineered Features:')
for feature in engineered_features:
    print('-', feature)
print('\nDay 15 Feature Engineering Complete!')